In [ ]:
import zipfile
import os

zip_path = '/content/CS117 Project Images-20261117T004451Z-1-001.zip'
extract_path = '/content/CS117_Project_Images'

# Unzip the file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Successfully unzipped to: {extract_path}")

# List the contents of the extracted directory
print("\nExtracted contents:")
for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = ' ' * 4 * (level + 1)
    for file in files:
        print(f"{sub_indent}{file}")

In [ ]:
!pip install geoclip

In [3]:
from pathlib import Path

directory = Path("/content/CS117_Project_Images")

# Get a list of Path objects
file_paths = [p for p in directory.rglob("*") if p.is_file()]

In [6]:
from geoclip import GeoCLIP
import torch

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GeoCLIP(from_pretrained=True)
model.to(device)


top_pred_gps, top_pred_prob = model.predict(image_path, top_k=5)

print("Top 5 GPS Predictions")
print("=====================")
for i in range(5):
    lat, lon = top_pred_gps[i]
    print(f"Prediction {i+1}: ({lat:.6f}, {lon:.6f})")
    print(f"Probability: {top_pred_prob[i]:.6f}")
    print("")

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Top 5 GPS Predictions
Prediction 1: (41.019779, 28.972610)
Probability: 0.014374

Prediction 2: (41.018944, 28.972546)
Probability: 0.014083

Prediction 3: (41.018440, 28.971836)
Probability: 0.013375

Prediction 4: (41.017857, 28.974594)
Probability: 0.013352

Prediction 5: (41.017826, 28.974014)
Probability: 0.013310



In [19]:
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS

def get_gps_decimal(image_path):
    def convert_to_degrees(value):
        d = float(value[0])
        m = float(value[1])
        s = float(value[2])
        return d + (m / 60.0) + (s / 3600.0)

    try:
        img = Image.open(image_path)
        exif_data = img._getexif()
        if not exif_data:
            print("No EXIF metadata found.")
            return None

        gps_info = {}
        for tag, value in exif_data.items():
            decoded = TAGS.get(tag, tag)
            if decoded == "GPSInfo":
                for t in value:
                    sub_decoded = GPSTAGS.get(t, t)
                    gps_info[sub_decoded] = value[t]

        if "GPSLatitude" in gps_info and "GPSLongitude" in gps_info:
            lat = convert_to_degrees(gps_info["GPSLatitude"])
            if gps_info.get("GPSLatitudeRef", "N") != "N":
                lat = -lat

            lon = convert_to_degrees(gps_info["GPSLongitude"])
            if gps_info.get("GPSLongitudeRef", "E") != "E":
                lon = -lon

            # print(f"Latitude (Decimal): {lat:.6f}")
            # print(f"Longitude (Decimal): {lon:.6f}")
            return lat, lon
        else:
            print("No GPS metadata found in EXIF.")
            return None
    except Exception as e:
        print(f"Error processing image: {e}")
        return None

# Example usage:
# get_gps_decimal(file_paths[0])

In [10]:
get_gps_decimal(image_path)

Latitude (Decimal): 41.017450
Longitude (Decimal): 28.985611


(41.01745, 28.985611111111112)

In [25]:
import json
import torch

def get_all_locations_top5_json(paths_list):
    results = []

    # Ensure model is on the correct device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    for p in paths_list:
        path_str = str(p)

        # 1. Get actual GPS coordinates from metadata
        actual_gps = get_gps_decimal(path_str)
        actual_lat, actual_lon = actual_gps if actual_gps else (None, None)

        # 2. Get top 5 predictions using GeoCLIP
        predictions_list = []
        try:
            top_pred_gps, top_pred_prob = model.predict(path_str, top_k=5)
            for i in range(len(top_pred_gps)):
                lat, lon = float(top_pred_gps[i][0]), float(top_pred_gps[i][1])
                prob = float(top_pred_prob[i])
                predictions_list.append({
                    "latitude": lat,
                    "longitude": lon,
                    "probability": prob
                })
        except Exception as e:
            pass

        results.append({
            "file_path": path_str,
            "actual_gps": {
                "latitude": actual_lat,
                "longitude": actual_lon
            },
            "top_5_predictions": predictions_list
        })

    return results

# To run and inspect on a subset:
# print(get_all_locations_top5_json(file_paths[:3]))

In [28]:
output = get_all_locations_top5_json(file_paths)

No GPS metadata found in EXIF.
No GPS metadata found in EXIF.


In [29]:
with open(r"/content/cs117_images_gps.json", "w") as f:
  json.dump(output, f, indent=4)
  print("Done!")

Done!


In [35]:
for i in range(331):
  print(output[i]["actual_gps"])

{'latitude': 41.01745, 'longitude': 28.985611111111112}
{'latitude': 37.80864166666667, 'longitude': -122.47539166666667}
{'latitude': 40.750277777777775, 'longitude': -73.96746944444445}
{'latitude': 31.58820833333333, 'longitude': 74.31051111111111}
{'latitude': -33.88967777777778, 'longitude': 151.18033888888888}
{'latitude': -33.891025, 'longitude': 151.28215833333334}
{'latitude': 30.263633333333335, 'longitude': -97.75106111111111}
{'latitude': 41.00647222222222, 'longitude': 28.97944722222222}
{'latitude': 37.81746944444445, 'longitude': -122.47429166666667}
{'latitude': 31.588247222222222, 'longitude': 74.31109166666667}
{'latitude': 41.00854722222222, 'longitude': 28.979977777777776}
{'latitude': 37.77958888888889, 'longitude': -122.49353333333333}
{'latitude': -33.889202777777776, 'longitude': 151.17971666666665}
{'latitude': 37.806411999999995, 'longitude': -122.447984}
{'latitude': 33.644755555555555, 'longitude': -117.84357222222222}
{'latitude': -33.86059166666667, 'longi

In [36]:
import math

def check_predictions_within_radius(data_list, max_distance_miles=5.0):
    def haversine_distance(coord1, coord2):
        # Coordinates in decimal degrees
        lat1, lon1 = coord1
        lat2, lon2 = coord2

        if None in (lat1, lon1, lat2, lon2):
            return float('inf')

        # Mean radius of Earth in miles
        R = 3958.8

        phi1 = math.radians(lat1)
        phi2 = math.radians(lat2)
        delta_phi = math.radians(lat2 - lat1)
        delta_lambda = math.radians(lon2 - lon1)

        a = math.sin(delta_phi / 2.0) ** 2 + \
            math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2.0) ** 2

        c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
        return R * c

    accuracy_dict = {}

    for item in data_list:
        path = item.get("file_path")
        actual = item.get("actual_gps", {})
        actual_coords = (actual.get("latitude"), actual.get("longitude"))

        is_within_limit = False

        # Check if actual GPS exists
        if actual_coords[0] is not None and actual_coords[1] is not None:
            for pred in item.get("top_5_predictions", []):
                pred_coords = (pred.get("latitude"), pred.get("longitude"))
                distance = haversine_distance(actual_coords, pred_coords)
                if distance <= max_distance_miles:
                    is_within_limit = True
                    break

        accuracy_dict[path] = is_within_limit

    return accuracy_dict

# Example usage with the existing 'output' variable:
# evaluation_results = check_predictions_within_radius(output)
# print(list(evaluation_results.items())[:5])

In [37]:
acc_dict = check_predictions_within_radius(output)

In [41]:
sum(acc_dict.values())/len(acc_dict.values())

0.5317220543806647